# 06 · 官方 EVSoC 工具与手势模型接入

本节按 TinyML 手册 3.4 复现工具生成和官方 Makefile 编译。默认读取实际执行报告；完整双链路硬件仍在集成，不能将本节成功解释为上板成功。

## 1. 找到项目和真实报告

`Path` 让 Notebook 在项目根目录或 notebooks 目录启动时都能定位文件。

In [1]:
from pathlib import Path
import json
root = Path.cwd()
if not (root / "MAIN.md").exists():
    root = root.parent
reports = root / "docs/05_experiments/v0.5"
generation = json.loads((reports / "official_generation.json").read_text())
build = json.loads((reports / "official_build.json").read_text())
assert generation["model_array_byte_exact"]
print(generation["model_sha256"])

7891518a9b70ec6b3be8649123651780ce87ede1e69a9a49c394c17d203a0baa


## 2. 读取官方工具生成的加速配置

参数来自未修改的官方生成器，而非手写替代文件。4×4 是待综合测量的初始候选；cache/FIFO 深度由模型分析更新。

In [2]:
keys = ["AXI_DW", "CONV_DEPTHW_MODE", "CONV_DEPTHW_STD_IN_PARALLEL", "CONV_DEPTHW_STD_OUT_PARALLEL", "FC_MODE", "RESHAPE_MODE", "TINYML_CACHE", "CACHE_DEPTH"]
{k: generation["parameters"][k] for k in keys}

{'AXI_DW': '128',
 'CONV_DEPTHW_MODE': 'STANDARD',
 'CONV_DEPTHW_STD_IN_PARALLEL': 4,
 'CONV_DEPTHW_STD_OUT_PARALLEL': 4,
 'FC_MODE': 'STANDARD',
 'RESHAPE_MODE': 'STANDARD',
 'TINYML_CACHE': 'ENABLE',
 'CACHE_DEPTH': '1024'}

## 3. 检查官方 Makefile 构建证据

返回值 0 和 ELF 哈希证明编译链接完成。此 BSP 仍来自官方 HyperRAM 示例，最终 DDR3 集成必须重新生成匹配 BSP。text/data/bss 是固件大小，不是 CNN 参数量；bss 包含默认 2 MiB 栈。

In [3]:
assert build["make_return_code"] == 0
print(build["elf_sha256"])
print(build["size_output"])
print(build["bsp_source"])
assert build["target_run_verified"] is False

b7889b9da8efb41862dd6faacde94fb8331f9b4e14ef60e9f30efd3900f67b5f
   text	   data	    bss	    dec	    hex	filename
 780396	  77078	2365704	3223178	 312e8a	C:\Users\SteLl1a\Desktop\CNN-Tutorial-FPGA\artifacts\evsoc-gesture-r1\embedded_sw\SapphireSoc\software\standalone\evsoc_tinyml_ypd\build\evsoc_tinyml_ypd.elf

copied official EVSoC HyperRAM demo; not final integrated DDR3 BSP


## 4. 在本机重新运行

在 Efinity 环境使用 `scripts/generate_official_tinyml.py --output artifacts/<新目录>`。FPGA 副本依次运行 `prepare_evsoc_gesture.py`、`adapt_evsoc_static.py`、`build_evsoc_application.py`。它们保留官方源文件快照、BSP 相对路径和哈希，且不下载硬件。

输入为 64×64 灰度 INT8，输出为 [1,3]；原 YOLO 框解码不能用于这三个 logits。静态自检通过后再接摄像头，步骤详见 [双链路设计](../docs/04_deployment/evsoc_dual_pipeline.md)。

## 语法小结

- `json.loads`：把 JSON 文本转为字典。
- `assert`：条件不满足即停止，避免把缺失证据当作通过。
- `None`：尚未测量；报告中的 arena_used_bytes 不应拿预分配容量填充。